# 4. Feature Engineering

## Purpose
Improve model performance by creating new features that better represent
the underlying patterns in the data. Feature engineering is deliberately
placed after baseline modelling so that SHAP values from the baseline
model can inform which features are worth engineering.

## Inputs
- `data/raw/` — raw data before preprocessing
- `config.yaml` — feature engineering settings, tuning model selection
- `artifacts/baseline_trainer.pkl` — baseline trainer for benchmark comparison
- `artifacts/preprocessor.pkl` — fitted preprocessing pipeline

## Outputs
- `data/interim/` — engineered features before preprocessing
- `data/processed/` — final selected features after preprocessing and
  two-stage feature selection (overwrites notebook 2 output)
- `artifacts/preprocessor.pkl` — updated fitted preprocessor
- `artifacts/selected_features.pkl` — list of selected feature names
- `artifacts/feature_engineering_trainer.pkl` — trainer state for comparison

## Decisions for the user
- Edit `src/features/build_features.py` to add domain-specific features —
  this is the most impactful form of feature engineering
- Enable automated feature engineering in `config.yaml` selectively:
  - `interactions` and `polynomial` — most useful for linear models,
    tree models find these naturally through splits
  - `binning` — useful for non-monotonic numeric relationships
  - `aggregations` — useful when group statistics carry signal
- Set `feature_selection.mi_percentile` — controls how aggressively
  mutual information filters features before SHAP refinement
- Set `feature_selection.shap_percentile` — controls final feature count

## Important note
Always compare the engineered CV score against the baseline before
proceeding. If engineered score < baseline score, revert feature
engineering flags to false in `config.yaml` and rely on domain
features in `build_features.py` instead.

In [1]:
import pandas as pd
import numpy as np
import joblib
import copy

from src.config.settings import load_config
from src.config.paths import (
    RAW_DATA_DIR,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    ARTIFACTS_DIR,
    BASELINE_MODELS_DIR
)
from src.data.load_data import load_raw_data
from src.features.feature_engineering import run_feature_engineering
from src.features.build_features import add_domain_features
from src.features.feature_selector import run_feature_selection
from src.features.pipeline import FeatureEngineeringPipeline
from src.models.training.trainer import ModelTrainer
from src.models.training.model_registry import MODEL_REGISTRY
from src.models.training.metrics import extract_primary_metric

c:\Users\kigr\anaconda3\envs\ml-template\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load config

In [2]:
config = load_config()

TARGET = config["target"]
DROP_COLS = config["drop_columns"]
SPLIT_CONFIG = config["split"]
PIPELINE_CONFIG = config["pipeline"]
BASELINE_CONFIG = config["baseline"]
TUNING_CONFIG = config["tuning"]

print(f"Target:           {TARGET}")
print(f"Models to tune:   {TUNING_CONFIG['models_to_tune']}")

Target:           Survived
Models to tune:   ['xgboost_classifier']


## 2. Load raw data
Raw data is reloaded and split using the same parameters as notebook 2
to ensure identical train/test splits.

In [3]:
from sklearn.model_selection import train_test_split

df = load_raw_data()
X_raw = df.drop(columns=[TARGET] + DROP_COLS)
y = df[TARGET]

stratify_col = y if SPLIT_CONFIG["stratify"] else None

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=SPLIT_CONFIG["test_size"],
    random_state=SPLIT_CONFIG["random_state"],
    stratify=stratify_col
)

print(f"X_train_raw shape: {X_train_raw.shape}")
print(f"X_test_raw shape:  {X_test_raw.shape}")

X_train_raw shape: (712, 10)
X_test_raw shape:  (179, 10)


## 3. Domain-specific features
Edit `src/features/build_features.py` to add features based on domain
knowledge. This is a no-op by default — safe to run on any dataset.

In [4]:
X_train_domain = add_domain_features(X_train_raw)
X_test_domain = add_domain_features(X_test_raw)

print(f"Shape after domain features - train: {X_train_domain.shape}, "
      f"test: {X_test_domain.shape}")

Shape after domain features - train: (712, 10), test: (179, 10)


## 4. Automated feature engineering
All automated steps are controlled by `config.yaml` and disabled by
default. Enable selectively based on your model type and dataset size.
Review the shape before and after to understand how many features were added.

In [5]:
X_train_engineered = run_feature_engineering(X_train_domain, config)
X_test_engineered = run_feature_engineering(X_test_domain, config)

print(f"Shape after automated engineering - train: {X_train_engineered.shape}, "
      f"test: {X_test_engineered.shape}")

Added 10 interaction features.
Added 15 binned features.


C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project

Added 10 interaction features.
Added 15 binned features.
Shape after automated engineering - train: (712, 35), test: (179, 35)


## 5. Save interim data
Engineered features are saved before preprocessing so the engineering
step does not need to be rerun if preprocessing settings change.

In [6]:
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

X_train_engineered.to_csv(INTERIM_DATA_DIR / "X_train_interim.csv", index=False)
X_test_engineered.to_csv(INTERIM_DATA_DIR / "X_test_interim.csv", index=False)
y_train.to_csv(INTERIM_DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(INTERIM_DATA_DIR / "y_test.csv", index=False)

print(f"Interim data saved to {INTERIM_DATA_DIR}")

Interim data saved to C:\ML\ML Workflow\Projects\Project Name Template\data\interim


## 6. Initialise preprocessor
A fresh unfitted preprocessor is initialised for use in the benchmark
comparison and feature selection steps below.

In [7]:
preprocessor = FeatureEngineeringPipeline(config=PIPELINE_CONFIG)

## 7. Benchmark comparison
The first model from `tuning.models_to_tune` in `config.yaml` is run
with the same CV setup as the baseline.
If the engineered score is lower than the baseline, revert the feature
engineering flags in `config.yaml`.

In [8]:
benchmark_model_name = TUNING_CONFIG["models_to_tune"][0]

trainer = ModelTrainer(
    model_registry=MODEL_REGISTRY,
    random_state=BASELINE_CONFIG["random_state"]
)

trainer.fit(
    X=X_train_engineered,
    y=y_train.values,
    preprocessor=preprocessor,
    cv=BASELINE_CONFIG["cv_folds"],
    run_all=False,
    model_subset=[benchmark_model_name]
)

Task detected: classification

Training: xgboost_classifier


C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project


Training complete.


## 8. Compare with baseline
Compares the engineered CV score directly against the baseline CV score.
A positive improvement means the new features are adding signal.
A negative improvement means they are adding noise — revert and investigate.

In [9]:
baseline_trainer = joblib.load(ARTIFACTS_DIR / "baseline_trainer.pkl")
baseline_score = baseline_trainer.leaderboard_[
    baseline_trainer.leaderboard_["model_name"] == benchmark_model_name
]["primary_metric_value"].values[0]

engineered_score = trainer.leaderboard_.iloc[0]["primary_metric_value"]
metric_name = trainer.leaderboard_.iloc[0]["primary_metric_name"]

print(f"\nBenchmark model: {benchmark_model_name}")
print(f"Baseline CV {metric_name}:    {baseline_score:.4f}")
print(f"Engineered CV {metric_name}:  {engineered_score:.4f}")
print(f"Improvement:               {engineered_score - baseline_score:+.4f}")


Benchmark model: xgboost_classifier
Baseline CV roc_auc:    0.8829
Engineered CV roc_auc:  0.8795
Improvement:               -0.0033


## 9. Two-stage feature selection
Stage 1 — mutual information pre-filter: fast removal of obviously
irrelevant features using statistical dependence with the target.
Stage 2 — SHAP refinement: precise, model-aware selection using
mean absolute SHAP values from the fitted benchmark model.
The final selection is the intersection of both stages.

In [10]:
# Get preprocessed training data for feature selection
fold_preprocessor = copy.deepcopy(preprocessor)
X_train_preprocessed = fold_preprocessor.fit_transform(
    X_train_engineered, y_train
)

# Get the fitted benchmark model
benchmark_model = trainer.models_[benchmark_model_name]

selected_features, selection_metadata = run_feature_selection(
    X_train=X_train_preprocessed,
    y_train=y_train,
    model=benchmark_model,
    task=trainer.task_,
    config=config
)

print(f"\nFinal selected features: {len(selected_features)}")

C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project Name Template\src\features\type_detection.py:125: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(non_null, errors="coerce")
C:\ML\ML Workflow\Projects\Project

MI filter: 131 → 65 features (top 50%)
SHAP filter: 131 → 104 features (top 80%)
Final selection (intersection): 57 features

Final selected features: 57


## 10. SHAP feature importance
Review the top features by SHAP importance. These are the features
driving the model's predictions — use them to inform further domain
feature engineering in `build_features.py`.

In [11]:
print("\nTop 20 features by SHAP importance:")
selection_metadata["shap_scores"].head(20)


Top 20 features by SHAP importance:


Sex__male          1.325047
Fare               0.357102
Pclass             0.349634
Pclass_x_Age       0.325743
Pclass_x_Fare      0.324935
Age_x_Fare         0.312005
Cabin__hash_25     0.271793
Age                0.240525
Embarked__S        0.171661
Sex__female        0.168159
Pclass_binned      0.108812
Ticket__hash_24    0.102673
SibSp_x_Fare       0.075942
Age_x_SibSp        0.075633
Ticket__hash_11    0.073617
Embarked__C        0.069595
Name__hash_5       0.067765
Pclass_x_SibSp     0.064444
SibSp_x_Parch      0.059059
Name__hash_11      0.054675
dtype: float32

## 11. Re-run Benchmark with Selected Features Only

In [12]:
X_train_selected = X_train_preprocessed[selected_features]
X_test_preprocessed = fold_preprocessor.transform(X_test_engineered)
X_test_selected = X_test_preprocessed[selected_features]

# Save selected feature names for downstream notebooks
joblib.dump(selected_features, ARTIFACTS_DIR / "selected_features.pkl")
print(f"Selected features saved to {ARTIFACTS_DIR / 'selected_features.pkl'}")

Selected features saved to C:\ML\ML Workflow\Projects\Project Name Template\artifacts\selected_features.pkl


C:\ML\ML Workflow\Projects\Project Name Template\src\features\pipeline.py:188: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}__hash_{j}"] = hashed[:, j]
C:\ML\ML Workflow\Projects\Project Name Template\src\features\pipeline.py:188: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}__hash_{j}"] = hashed[:, j]
C:\ML\ML Workflow\Projects\Project Name Template\src\features\pipeline.py:188: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has p

## 12. Save final processed data with selected features
The selected features are saved to `data/processed/` overwriting the
output from notebook 2. All downstream notebooks use this final
feature set.

In [13]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

X_train_selected_df = pd.DataFrame(X_train_selected, columns=selected_features)
X_test_selected_df = pd.DataFrame(X_test_selected, columns=selected_features)

X_train_selected_df.to_csv(PROCESSED_DATA_DIR / "X_train.csv", index=False)
X_test_selected_df.to_csv(PROCESSED_DATA_DIR / "X_test.csv", index=False)

print(f"Final processed data saved to {PROCESSED_DATA_DIR}")
print(f"Final feature count: {len(selected_features)}")

Final processed data saved to C:\ML\ML Workflow\Projects\Project Name Template\data\processed
Final feature count: 57


## 13. Save Feature Engineering Artifacts

In [14]:
joblib.dump(fold_preprocessor, ARTIFACTS_DIR / "preprocessor.pkl")
joblib.dump(trainer, ARTIFACTS_DIR / "feature_engineering_trainer.pkl")
print("Feature engineering artifacts saved.")

Feature engineering artifacts saved.
